# Polymorph Pipeline — Raman Ground Truth → Segmentation → Classification

**Companion notebook for** *"Raman-Grounded Multimodal Sensing of CaCO\u2083\nPolymorphs during Microfluidic Biomineralization"* (Filanoski & Erickson,
ACS Sensors, in preporation).

This notebook walks through the full polymorph identification pipeline on
four representative channels \u2014 `set2/c3`, `set2/c6`, `set4/c8`, and
`set5/c8` \u2014 spanning all three polymorph classes. Sets 1, 2, and 4
contribute calcite and early-stage vaterite; sets 3 and 5 contribute
late-stage vaterite.

## Pipeline

1. Load Raman ground-truth labels from a pre-built parquet
2. Plot the raw Raman spectra grouped by polymorph class
3. Overlay Raman point locations on the brightfield image
4. Color the points by their Raman-derived polymorph label
5. Segment crystals with Cellpose
6. Color the resulting masks by Raman ground truth (label propagation)
7. Predict polymorph class for every instance with a 5-channel EfficientNet-B0
8. Compare classifier predictions against the Raman ground truth

## Data

All polymorph labels were extracted from the Raman spectra and quality
checked. Raman stage coordinates were aligned to the PNG image coordinate
system and quality checked. The example dataset is in
`zenodo_upload/publication_data/`; edit `PUB_DIR` in the setup cell to
point at a local copy.

## Requirements

Colab default. GPU recommended for Cellpose + classifier inference
(works on CPU, just slower). Runtime: a few minutes end-to-end.


## 1. Setup and data loading

Mount Drive, load the consolidated parquet, and define the polymorph color
scheme used throughout the figures. The parquet contains all 550 Raman-labeled
training points across the 5 sets; raw spectra and PNGs are embedded only for
the 4 example channels.


In [ ]:
# ============================================================
# SETUP AND DATA LOADING
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image

# ------------------------------------------------------------
# *** USER CONFIG ***
# ------------------------------------------------------------
PUB_DIR = Path(
    '/content/drive/MyDrive/polymorph_github_staging/zenodo_upload/publication_data'
)

# class colors (consistent with manuscript figures)
CLASS_COLORS = {
    'calcite':        '#2166ac',   # blue
    'early_vaterite': '#d6604d',   # red
    'late_vaterite':  '#50c878',   # green
}
CLASS_LABELS = {
    'calcite':        'Calcite',
    'early_vaterite': 'Early-stage vaterite',
    'late_vaterite':  'Late-stage vaterite',
}
# ------------------------------------------------------------

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    14,
    'axes.labelsize':    14,
    'xtick.labelsize':   13,
    'ytick.labelsize':   13,
    'legend.fontsize':   12,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
df = pd.read_parquet(PUB_DIR / 'raman_data.parquet')

# summary
print(f'Data directory: {PUB_DIR}')
print(f'Total training points: {len(df)}\n')
print('By set × polymorph:')
print(df.groupby(['set', 'polymorph']).size().unstack(fill_value=0))
print(f'\nExample channels (with raw spectra + images): '
      f'{df["wavenumber_cm"].notna().sum()} points across '
      f'{df.loc[df["wavenumber_cm"].notna(), ["set","channel"]].drop_duplicates().shape[0]} channels')

## 2. Raman spectra by polymorph class

For each of the 4 example channels, plot the raw Raman spectra grouped by
polymorph class. Calcite shows a sharp \u03BD\u2081 peak near 1086 cm\u207B\u00B9;
vaterite shows a characteristic doublet near 1075/1090 cm\u207B\u00B9. These
spectral fingerprints are the basis of the ground-truth labels used downstream.


In [ ]:
# ============================================================
# FIGURE — Raman spectra: channels (cols) × {calcite, vaterite} (rows)
# ============================================================
PUB_DIR = Path('/content/drive/MyDrive/polymorph_github_staging/zenodo_upload/publication_data')

df    = pd.read_parquet(PUB_DIR / 'raman_data.parquet')
df_ex = df[df['wavenumber_cm'].notna()]
groups = list(df_ex.groupby(['set', 'channel'], sort=False))

SPEC_XMIN, SPEC_XMAX = 200, 1100
row_groups = [
    ('Calcite',  ['calcite']),
    ('Vaterite', ['early_vaterite', 'late_vaterite']),
]

fig, axes = plt.subplots(2, len(groups),
                         figsize=(4.5 * len(groups), 6),
                         sharex=True)

for j, ((set_name, channel), pts) in enumerate(groups):
    for i, (row_title, classes) in enumerate(row_groups):
        ax = axes[i, j]
        sub = pts[pts['polymorph'].isin(classes)]
        for _, row in sub.iterrows():
            x = np.array(row['wavenumber_cm'])
            y = np.array(row['intensity_counts'])
            m = (x >= SPEC_XMIN) & (x <= SPEC_XMAX)
            ax.plot(x[m], y[m],
                    color=CLASS_COLORS[row['polymorph']],
                    linewidth=0.7, alpha=0.5)

        ax.set_xlim(SPEC_XMIN, SPEC_XMAX)
        if i == 0:
            ax.set_title(f'{set_name}/{channel}', fontsize=13)
        if j == 0:
            ax.set_ylabel(f'{row_title}\nIntensity (CCD cts)', fontsize=11)
        if i == 1:
            ax.set_xlabel(r'Raman shift (cm$^{-1}$)')

plt.tight_layout()
plt.show()

## 3. Raman point locations on the brightfield image

Each yellow dot is one Raman acquisition point, projected onto the brightfield
image using the channel's pixel calibration. Coordinates come directly from
the parquet (`x_px`, `y_px`) \u2014 already aligned to image coordinates.


In [ ]:
# ============================================================
# FIGURE — Raman point locations on RA image (unclassified)
# ============================================================
PUB_DIR = Path('/content/drive/MyDrive/polymorph_github_staging/zenodo_upload/publication_data')

df    = pd.read_parquet(PUB_DIR / 'raman_data.parquet')
df_ex = df[df['wavenumber_cm'].notna()]
groups = list(df_ex.groupby(['set', 'channel'], sort=False))

fig, axes = plt.subplots(1, len(groups), figsize=(18, 5))

for ax, ((set_name, channel), pts) in zip(axes, groups):
    img = Image.open(PUB_DIR / 'images' / pts['image_ra'].iloc[0])
    ax.imshow(img)
    ax.scatter(pts['x_px'], pts['y_px'],
               s=80, marker='o',
               facecolor='yellow', edgecolor='black', linewidth=1.2,
               zorder=5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'{set_name} / {channel}', fontsize=13)

plt.tight_layout()
plt.show()

## 4. Raman points colored by polymorph label

Same Raman points as above, now colored by the polymorph label extracted
from each point's spectrum. This is the ground-truth signal that anchors
the segmentation and classification steps that follow.


In [ ]:
# ============================================================
# FIGURE — Spatial overlay of Raman points on RA image (classified)
# ============================================================
PUB_DIR = Path('/content/drive/MyDrive/polymorph_github_staging/zenodo_upload/publication_data')

df    = pd.read_parquet(PUB_DIR / 'raman_data.parquet')
df_ex = df[df['wavenumber_cm'].notna()]
groups = list(df_ex.groupby(['set', 'channel'], sort=False))

fig, axes = plt.subplots(1, len(groups), figsize=(18, 5))

for ax, ((set_name, channel), pts) in zip(axes, groups):
    img = Image.open(PUB_DIR / 'images' / pts['image_ra'].iloc[0])
    ax.imshow(img)

    for cls, color in CLASS_COLORS.items():
        m = pts['polymorph'] == cls
        if not m.any():
            continue
        ax.scatter(pts.loc[m, 'x_px'], pts.loc[m, 'y_px'],
                   s=80, marker='o',
                   facecolor=color, edgecolor='white', linewidth=1.5,
                   label=CLASS_LABELS[cls], zorder=5)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'{set_name} / {channel}', fontsize=13)

handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
           markeredgecolor='white', markersize=10, label=CLASS_LABELS[k])
    for k, c in CLASS_COLORS.items()
]
fig.legend(handles=handles, loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, -0.02), frameon=False)

plt.tight_layout()
plt.show()

## 5. Cellpose instance segmentation

Run Cellpose on each channel's brightfield + polarized-light pair to identify
individual crystal instances. Each colored region is one detected crystal.
The model (`cellpose_5set`) was retrained on this dataset; inference uses
the optimal parameter set from the validation sweep.


In [ ]:
# ============================================================
# CELL 02 — Cellpose segmentation overlay (4 example channels)
# ============================================================
import subprocess, sys
try:
    from cellpose.models import CellposeModel
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cellpose==3.1.0'])
    from cellpose.models import CellposeModel

import cv2
import matplotlib.cm as cm
from pathlib import Path

# ------------------------------------------------------------
# *** USER CONFIG — point at trained model ***
# ------------------------------------------------------------
MODEL_PATH = '/content/drive/MyDrive/polymorph_github_staging/zenodo_upload/models/cellpose_5set'
# ------------------------------------------------------------

CP_DIAMETER  = 40
CP_FLOW      = 0.6
CP_CELLPROB  = 0.0
CP_MIN_SIZE  = 50
IMG_MAX_SIDE = 1024

assert Path(MODEL_PATH).exists(), f'MODEL_PATH does not exist: {MODEL_PATH}'

def load_norm(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE).astype(np.float32)
    h, w = img.shape
    s = IMG_MAX_SIDE / max(h, w)
    if s < 1.0:
        img = cv2.resize(img, (int(round(w*s)), int(round(h*s))), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(img, 1), np.percentile(img, 99.8)
    return np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)

def colorize(inst, seed=42):
    rng = np.random.default_rng(seed)
    n = int(inst.max())
    if n == 0:
        return np.zeros((*inst.shape, 3), dtype=np.uint8)
    cmap = cm.get_cmap('gist_ncar')
    vals = np.linspace(0.05, 0.95, n); rng.shuffle(vals)
    colors = np.zeros((n+1, 3), dtype=np.uint8)
    for i, v in enumerate(vals):
        r, g, b, _ = cmap(v); colors[i+1] = [int(r*255), int(g*255), int(b*255)]
    return colors[inst]

def overlay(ra_norm, inst, alpha=0.55):
    h, w = ra_norm.shape
    if inst.shape != (h, w):
        inst = cv2.resize(inst.astype(np.int32), (w, h), interpolation=cv2.INTER_NEAREST).astype(np.int32)
    base = (np.stack([ra_norm]*3, axis=-1) * 255).astype(np.float32)
    col  = colorize(inst).astype(np.float32)
    mask = (inst > 0)[..., None]
    return np.clip(np.where(mask, (1-alpha)*base + alpha*col, base), 0, 255).astype(np.uint8)

# ------------------------------------------------------------
# Run inference on the 4 example channels
# ------------------------------------------------------------
df_ex = df[df['wavenumber_cm'].notna()]
example_channels = df_ex[['set', 'channel', 'image_ra', 'image_pol']].drop_duplicates().reset_index(drop=True)

print('Loading Cellpose model...')
model = CellposeModel(gpu=True, pretrained_model=MODEL_PATH)
print('Done.\n')

fig, axes = plt.subplots(1, len(example_channels), figsize=(18, 5))

for ax, (_, row) in zip(axes, example_channels.iterrows()):
    ra  = load_norm(PUB_DIR / 'images' / row['image_ra'])
    pol = load_norm(PUB_DIR / 'images' / row['image_pol'])
    if pol.shape != ra.shape:
        pol = cv2.resize(pol, (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
    stack = np.stack([ra, pol], axis=0)

    masks, _, _ = model.eval(
        [stack],
        diameter=CP_DIAMETER,
        flow_threshold=CP_FLOW,
        cellprob_threshold=CP_CELLPROB,
        channels=[1, 2],
        normalize=False,
        min_size=CP_MIN_SIZE,
    )
    inst = masks[0].astype(np.int32)

    ax.imshow(overlay(ra, inst))
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{row["set"]} / {row["channel"]}  n={int(inst.max())}', fontsize=13)

plt.suptitle('Cellpose segmentation — cellpose_5set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Label propagation: Cellpose masks colored by Raman ground truth

For each Raman point, find the Cellpose instance it falls inside and color
that instance by its polymorph label. Crystals without a Raman point stay
gray. This is how 86 spectroscopic labels become ~140 labeled crystal
instances \u2014 the training data for the classifier.


In [ ]:
# ============================================================
# CELL — Cellpose masks color-coded by Raman ground-truth labels
# ============================================================
df_ex = df[df['wavenumber_cm'].notna()]
example_channels = df_ex[['set', 'channel', 'image_ra', 'image_pol']].drop_duplicates().reset_index(drop=True)

SEARCH_RADIUS = 15  # px at full res — used to snap off-mask points to nearest instance

fig, axes = plt.subplots(1, len(example_channels), figsize=(18, 5))

for ax, (_, row) in zip(axes, example_channels.iterrows()):
    set_name, channel = row['set'], row['channel']

    # full-res RA for display + sizing reference
    ra_full = np.array(Image.open(PUB_DIR / 'images' / row['image_ra']).convert('L'))
    H_full, W_full = ra_full.shape

    # 1024-scaled RA + POL for cellpose
    ra  = load_norm(PUB_DIR / 'images' / row['image_ra'])
    pol = load_norm(PUB_DIR / 'images' / row['image_pol'])
    if pol.shape != ra.shape:
        pol = cv2.resize(pol, (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
    stack = np.stack([ra, pol], axis=0)

    masks, _, _ = model.eval(
        [stack],
        diameter=CP_DIAMETER, flow_threshold=CP_FLOW,
        cellprob_threshold=CP_CELLPROB, channels=[1, 2],
        normalize=False, min_size=CP_MIN_SIZE,
    )
    inst_small = masks[0].astype(np.int32)

    # upscale instance mask to full resolution (nearest-neighbor preserves IDs)
    inst = cv2.resize(inst_small, (W_full, H_full), interpolation=cv2.INTER_NEAREST)

    # raman points are in full-resolution pixel coords — use directly
    pts = df_ex[(df_ex['set'] == set_name) & (df_ex['channel'] == channel)].copy()
    pts['col'] = pts['x_px'].round().astype(int).clip(0, W_full - 1)
    pts['row'] = pts['y_px'].round().astype(int).clip(0, H_full - 1)
    pts['inst_id'] = inst[pts['row'].values, pts['col'].values]

    # for points on background, snap to nearest instance within search radius
    miss_idx = pts.index[pts['inst_id'] == 0]
    for idx in miss_idx:
        r0, c0 = pts.at[idx, 'row'], pts.at[idx, 'col']
        rr0, rr1 = max(0, r0 - SEARCH_RADIUS), min(H_full, r0 + SEARCH_RADIUS + 1)
        cc0, cc1 = max(0, c0 - SEARCH_RADIUS), min(W_full, c0 + SEARCH_RADIUS + 1)
        patch = inst[rr0:rr1, cc0:cc1]
        if patch.max() == 0:
            continue
        ys, xs = np.where(patch > 0)
        d = (ys - (r0 - rr0))**2 + (xs - (c0 - cc0))**2
        best = np.argmin(d)
        pts.at[idx, 'inst_id'] = int(patch[ys[best], xs[best]])

    inst_to_class = {int(p['inst_id']): p['polymorph']
                     for _, p in pts.iterrows() if p['inst_id'] > 0}

    # base = full-res grayscale RA, color labeled instances
    base = np.stack([ra_full]*3, axis=-1).astype(np.float32)
    overlay_img = base.copy()
    alpha = 0.55
    for iid, cls in inst_to_class.items():
        rgb = np.array([int(CLASS_COLORS[cls][i:i+2], 16) for i in (1, 3, 5)], dtype=np.float32)
        m = (inst == iid)
        overlay_img[m] = (1 - alpha) * base[m] + alpha * rgb

    ax.imshow(np.clip(overlay_img, 0, 255).astype(np.uint8))

    miss = pts[pts['inst_id'] == 0]
    if len(miss):
        ax.scatter(miss['col'], miss['row'], s=40, marker='x',
                   color='white', linewidth=1.5, zorder=6)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{set_name}/{channel}  '
                 f'({len(inst_to_class)}/{int(inst_small.max())} labeled)',
                 fontsize=13)

handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=c,
           markersize=12, label=CLASS_LABELS[k])
    for k, c in CLASS_COLORS.items()
]
handles.append(Line2D([0], [0], marker='x', color='black',
                      markersize=10, linestyle='', label='Raman point off mask'))
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Cellpose masks color-coded by Raman ground-truth label',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Classifier predictions on every instance

Run the trained EfficientNet-B0 polymorph classifier on every Cellpose
instance in each channel. Input is 5-channel: brightfield + binary mask +
polarized-light R/G/B. Output is one of three classes: calcite,
early-stage vaterite, or late-stage vaterite. This extends the labels
from the Raman-anchored subset to all detected crystals.


In [ ]:
# ============================================================
# CELL 03 — Polymorph classification overlay (4 example channels)
# ============================================================
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

# ------------------------------------------------------------
# *** USER CONFIG — point at trained classifier ***
# ------------------------------------------------------------
CLASSIFIER_PATH = '/content/drive/MyDrive/polymorph_github_staging/zenodo_upload/models/model5_3class_best.pth'
# ------------------------------------------------------------

CROP_SIZE = 224
PAD_UM    = 10.0

# classifier output order (must match training)
CLF_CLASSES = ['calcite', 'fresh_vaterite', 'old_vaterite']
# map classifier names → notebook polymorph names (for color lookup)
CLF_TO_POLY = {
    'calcite':        'calcite',
    'fresh_vaterite': 'early_vaterite',
    'old_vaterite':   'late_vaterite',
}

assert Path(CLASSIFIER_PATH).exists(), f'CLASSIFIER_PATH does not exist: {CLASSIFIER_PATH}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ------------------------------------------------------------
# Build EfficientNet-B0: 5-channel input, 3-class output
# ------------------------------------------------------------
def build_classifier():
    net = efficientnet_b0(weights=None)
    old_conv = net.features[0][0]
    new_conv = nn.Conv2d(5, old_conv.out_channels,
                         kernel_size=old_conv.kernel_size,
                         stride=old_conv.stride,
                         padding=old_conv.padding,
                         bias=False)
    net.features[0][0] = new_conv
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, 3)
    return net

clf = build_classifier()
state = torch.load(CLASSIFIER_PATH, map_location=device)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
clf.load_state_dict(state)
clf = clf.to(device).eval()
print('Classifier loaded.\n')

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_pol_rgb_norm(path):
    """Load POL as RGB, resize to 1024 max side, return per-channel normalized (R,G,B)."""
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    s = IMG_MAX_SIDE / max(h, w)
    if s < 1.0:
        rgb = cv2.resize(rgb, (int(round(w*s)), int(round(h*s))), interpolation=cv2.INTER_AREA)
    def _norm(c):
        c = c.astype(np.float32)
        lo, hi = np.percentile(c, 1), np.percentile(c, 99.8)
        return np.clip((c - lo) / (hi - lo + 1e-8), 0, 1)
    return _norm(rgb[:,:,0]), _norm(rgb[:,:,1]), _norm(rgb[:,:,2])

# ------------------------------------------------------------
# Run pipeline on the 4 example channels
# ------------------------------------------------------------
df_ex = df[df['wavenumber_cm'].notna()]
example_channels = df_ex[['set', 'channel', 'image_ra', 'image_pol']].drop_duplicates().reset_index(drop=True)

fig, axes = plt.subplots(1, len(example_channels), figsize=(18, 5))

for ax, (_, row) in zip(axes, example_channels.iterrows()):
    set_name, channel = row['set'], row['channel']

    # full-res RA for display
    ra_full = np.array(Image.open(PUB_DIR / 'images' / row['image_ra']).convert('L'))
    H_full, W_full = ra_full.shape

    # 1024-scaled inputs for cellpose + classifier
    ra      = load_norm(PUB_DIR / 'images' / row['image_ra'])
    pol_gs  = load_norm(PUB_DIR / 'images' / row['image_pol'])
    pol_r, pol_g, pol_b = load_pol_rgb_norm(PUB_DIR / 'images' / row['image_pol'])
    if pol_gs.shape != ra.shape:
        pol_gs = cv2.resize(pol_gs, (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
        pol_r  = cv2.resize(pol_r,  (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
        pol_g  = cv2.resize(pol_g,  (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
        pol_b  = cv2.resize(pol_b,  (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
    H, W = ra.shape

    # cellpose
    masks, _, _ = model.eval(
        [np.stack([ra, pol_gs], axis=0)],
        diameter=CP_DIAMETER, flow_threshold=CP_FLOW,
        cellprob_threshold=CP_CELLPROB, channels=[1, 2],
        normalize=False, min_size=CP_MIN_SIZE,
    )
    inst = masks[0].astype(np.int32)
    n_inst = int(inst.max())

    # padding in inference-space pixels (matches training: PAD_UM * scale_inf)
    img_w_um  = W_full / 4.289   # PNGs are scale 4.289 px/um at full res; inference scale ~1 px/um
    scale_inf = W / img_w_um
    pad_px    = int(round(PAD_UM * scale_inf))

    # classify each instance
    inst_to_class = {}
    crops, ids = [], []
    for iid in range(1, n_inst + 1):
        mask_inst = (inst == iid).astype(np.uint8)
        ys, xs = np.where(mask_inst)
        if len(ys) == 0:
            continue
        r0 = max(0, ys.min() - pad_px); r1 = min(H, ys.max() + pad_px)
        c0 = max(0, xs.min() - pad_px); c1 = min(W, xs.max() + pad_px)

        def _crop(arr2d):
            return cv2.resize(arr2d[r0:r1, c0:c1], (CROP_SIZE, CROP_SIZE),
                              interpolation=cv2.INTER_AREA)

        X = np.stack([
            _crop(ra),
            _crop(mask_inst.astype(np.float32)),
            _crop(pol_r), _crop(pol_g), _crop(pol_b),
        ], axis=0).astype(np.float32)
        crops.append(X)
        ids.append(iid)

    if crops:
        X_batch = torch.from_numpy(np.stack(crops, axis=0)).to(device)
        with torch.no_grad():
            logits = clf(X_batch)
            pred_idx = logits.argmax(dim=1).cpu().numpy()
        for iid, pi in zip(ids, pred_idx):
            inst_to_class[iid] = CLF_TO_POLY[CLF_CLASSES[pi]]

    # upscale mask to full res
    inst_full = cv2.resize(inst, (W_full, H_full), interpolation=cv2.INTER_NEAREST)

    # compose overlay
    base = np.stack([ra_full]*3, axis=-1).astype(np.float32)
    overlay_img = base.copy()
    alpha = 0.55
    for iid, cls in inst_to_class.items():
        rgb = np.array([int(CLASS_COLORS[cls][i:i+2], 16) for i in (1, 3, 5)], dtype=np.float32)
        m = (inst_full == iid)
        overlay_img[m] = (1 - alpha) * base[m] + alpha * rgb

    ax.imshow(np.clip(overlay_img, 0, 255).astype(np.uint8))
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{set_name}/{channel}  (n={len(inst_to_class)})', fontsize=13)

handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=c,
           markersize=12, label=CLASS_LABELS[k])
    for k, c in CLASS_COLORS.items()
]
fig.legend(handles=handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('EfficientNet-B0 polymorph predictions on Cellpose instances',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Classifier predictions vs. Raman ground truth

For the Raman-labeled subset, fill the instance with its ground-truth color
and outline it: green if the classifier agrees, red if it disagrees. The
title reports per-channel-aggregate accuracy.


In [ ]:
# ============================================================
# CELL — Classifier predictions vs Raman ground truth
# Fill = Raman GT class color, outline = correct (green) / incorrect (red)
# ============================================================
df_ex = df[df['wavenumber_cm'].notna()]
example_channels = df_ex[['set', 'channel', 'image_ra', 'image_pol']].drop_duplicates().reset_index(drop=True)

SEARCH_RADIUS = 15
CORRECT_COLOR = '#2ca02c'   # green
WRONG_COLOR   = '#d62728'   # red

fig, axes = plt.subplots(1, len(example_channels), figsize=(18, 5))

n_correct, n_wrong, n_total_labeled = 0, 0, 0

for ax, (_, row) in zip(axes, example_channels.iterrows()):
    set_name, channel = row['set'], row['channel']

    # full-res RA for display
    ra_full = np.array(Image.open(PUB_DIR / 'images' / row['image_ra']).convert('L'))
    H_full, W_full = ra_full.shape

    # 1024-scaled inputs
    ra      = load_norm(PUB_DIR / 'images' / row['image_ra'])
    pol_gs  = load_norm(PUB_DIR / 'images' / row['image_pol'])
    pol_r, pol_g, pol_b = load_pol_rgb_norm(PUB_DIR / 'images' / row['image_pol'])
    if pol_gs.shape != ra.shape:
        pol_gs = cv2.resize(pol_gs, (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
        pol_r  = cv2.resize(pol_r,  (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
        pol_g  = cv2.resize(pol_g,  (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
        pol_b  = cv2.resize(pol_b,  (ra.shape[1], ra.shape[0]), interpolation=cv2.INTER_AREA)
    H, W = ra.shape

    # cellpose
    masks, _, _ = model.eval(
        [np.stack([ra, pol_gs], axis=0)],
        diameter=CP_DIAMETER, flow_threshold=CP_FLOW,
        cellprob_threshold=CP_CELLPROB, channels=[1, 2],
        normalize=False, min_size=CP_MIN_SIZE,
    )
    inst = masks[0].astype(np.int32)

    img_w_um  = W_full / 4.289
    scale_inf = W / img_w_um
    pad_px    = int(round(PAD_UM * scale_inf))

    # classify each instance
    inst_to_pred = {}
    crops, ids = [], []
    for iid in range(1, int(inst.max()) + 1):
        m = (inst == iid).astype(np.uint8)
        ys, xs = np.where(m)
        if len(ys) == 0:
            continue
        r0 = max(0, ys.min() - pad_px); r1 = min(H, ys.max() + pad_px)
        c0 = max(0, xs.min() - pad_px); c1 = min(W, xs.max() + pad_px)
        def _crop(arr2d):
            return cv2.resize(arr2d[r0:r1, c0:c1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA)
        X = np.stack([_crop(ra), _crop(m.astype(np.float32)),
                      _crop(pol_r), _crop(pol_g), _crop(pol_b)], axis=0).astype(np.float32)
        crops.append(X); ids.append(iid)

    if crops:
        X_batch = torch.from_numpy(np.stack(crops, axis=0)).to(device)
        with torch.no_grad():
            preds = clf(X_batch).argmax(dim=1).cpu().numpy()
        for iid, pi in zip(ids, preds):
            inst_to_pred[iid] = CLF_TO_POLY[CLF_CLASSES[pi]]

    # upscale mask to full res
    inst_full = cv2.resize(inst, (W_full, H_full), interpolation=cv2.INTER_NEAREST)

    # raman ground truth → instance id
    pts = df_ex[(df_ex['set'] == set_name) & (df_ex['channel'] == channel)].copy()
    pts['col'] = pts['x_px'].round().astype(int).clip(0, W_full - 1)
    pts['row'] = pts['y_px'].round().astype(int).clip(0, H_full - 1)
    pts['inst_id'] = inst_full[pts['row'].values, pts['col'].values]

    miss_idx = pts.index[pts['inst_id'] == 0]
    for idx in miss_idx:
        r0, c0 = pts.at[idx, 'row'], pts.at[idx, 'col']
        rr0, rr1 = max(0, r0 - SEARCH_RADIUS), min(H_full, r0 + SEARCH_RADIUS + 1)
        cc0, cc1 = max(0, c0 - SEARCH_RADIUS), min(W_full, c0 + SEARCH_RADIUS + 1)
        patch = inst_full[rr0:rr1, cc0:cc1]
        if patch.max() == 0:
            continue
        ys, xs = np.where(patch > 0)
        d = (ys - (r0 - rr0))**2 + (xs - (c0 - cc0))**2
        pts.at[idx, 'inst_id'] = int(patch[ys[np.argmin(d)], xs[np.argmin(d)]])

    inst_to_gt = {int(p['inst_id']): p['polymorph']
                  for _, p in pts.iterrows() if p['inst_id'] > 0}

    # base = full-res grayscale
    base = np.stack([ra_full]*3, axis=-1).astype(np.float32)
    overlay_img = base.copy()
    alpha = 0.55

    # fill Raman-labeled instances with their GT color
    for iid, gt_cls in inst_to_gt.items():
        rgb = np.array([int(CLASS_COLORS[gt_cls][i:i+2], 16) for i in (1, 3, 5)], dtype=np.float32)
        m = (inst_full == iid)
        overlay_img[m] = (1 - alpha) * base[m] + alpha * rgb

    ax.imshow(np.clip(overlay_img, 0, 255).astype(np.uint8))

    # outline only the Raman-labeled instances: green if pred matches GT, red if not
    for iid, gt_cls in inst_to_gt.items():
        pred_cls = inst_to_pred.get(iid)
        if pred_cls is None:
            continue
        n_total_labeled += 1
        outline = CORRECT_COLOR if pred_cls == gt_cls else WRONG_COLOR
        if pred_cls == gt_cls: n_correct += 1
        else:                  n_wrong   += 1

        m = (inst_full == iid).astype(np.uint8)
        contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea).squeeze()
        if contour.ndim != 2:
            continue
        ax.plot(contour[:, 0], contour[:, 1],
                color=outline, linewidth=2.5, zorder=5)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{set_name}/{channel}', fontsize=13)

# legend
handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=c,
           markersize=12, label=CLASS_LABELS[k])
    for k, c in CLASS_COLORS.items()
]
handles.append(Line2D([0], [0], color=CORRECT_COLOR, linewidth=2.5, label='Classifier correct'))
handles.append(Line2D([0], [0], color=WRONG_COLOR,   linewidth=2.5, label='Classifier incorrect'))
fig.legend(handles=handles, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=10)

acc = n_correct / n_total_labeled if n_total_labeled else float('nan')
plt.suptitle(f'Classifier prediction vs. Raman ground truth — '
             f'{n_correct}/{n_total_labeled} correct ({acc:.1%})',
             fontsize=14, y=1.04)
plt.tight_layout()
plt.show()